# Trabajo Práctico Integrador

## Urban Flow


## Integrantes del grupo

- Completar con nombre y apellido.
- Completar con nombre y apellido.


## Sprint 1

Primer avance del proyecto: configuración del repositorio, estructura de carpetas y base del notebook para los próximos ejercicios.


## Configuración inicial

En esta primera entrega parcial se deja preparado el entorno de trabajo:

- Imports generales.
- Variables globales.
- Estructura de carpetas.
- Flujo básico de Git con `desactivar_git_push`.


In [ ]:
from urllib.request import urlretrieve
from datetime import datetime
from pathlib import Path
import re
import subprocess
import unicodedata

import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
desactivar_git_push = True
nombre_rama = "Sprint_1"
url_repositorio_https = ""
dataset_url = "https://raw.githubusercontent.com/HAD141/datasets/refs/heads/main/TrabajosPracticos/urban_flow/speeding_fines.csv"

repo_root = Path.cwd()
project_root = repo_root / "urban_flow"
raw_dir = project_root / "data" / "raw"
interim_dir = project_root / "data" / "interim"
processed_dir = project_root / "data" / "processed"
plots_dir = interim_dir / "plots"
raw_dataset_path = raw_dir / "speeding_fines.csv"


## Ejercicio 01

> Puntos: 1

Inicialización y configuración de la herramienta de versionado. Se trabaja sobre la rama `Sprint_1` y se crea la estructura pedida para el proyecto.


In [ ]:
for directory in [raw_dir, interim_dir, processed_dir, plots_dir]:
    directory.mkdir(parents=True, exist_ok=True)

for ruta in [project_root, raw_dir, interim_dir, processed_dir, plots_dir]:
    print(ruta.relative_to(repo_root))


In [ ]:
def ejecutar_git(args: list[str]) -> None:
    resultado = subprocess.run(["git", *args], capture_output=True, text=True)
    if resultado.stdout.strip():
        print(resultado.stdout.strip())
    if resultado.stderr.strip():
        print(resultado.stderr.strip())

ramas = subprocess.run(["git", "branch", "--list", nombre_rama], capture_output=True, text=True)
if ramas.stdout.strip():
    ejecutar_git(["checkout", nombre_rama])
else:
    ejecutar_git(["checkout", "-b", nombre_rama])


In [ ]:
print("Estado actual del repositorio:")
ejecutar_git(["status", "--short", "--branch"])


In [ ]:
def obtener_github_token() -> str:
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return ""

def push_si_corresponde() -> None:
    if desactivar_git_push:
        print("git push desactivado para esta ejecución.")
        return
    if not url_repositorio_https:
        print("No se configuró la URL del repositorio remoto.")
        return
    github_token = obtener_github_token()
    if not github_token:
        print("No se encontró GITHUB_TOKEN en los secrets de Colab.")
        return
    remote_url_con_token = url_repositorio_https.replace("https://", f"https://{github_token}@")
    subprocess.run(["git", "remote", "remove", "origin"], capture_output=True, text=True)
    subprocess.run(["git", "remote", "add", "origin", remote_url_con_token], capture_output=True, text=True)
    ejecutar_git(["push", "-u", "origin", nombre_rama])


In [ ]:
print("Comandos sugeridos para este primer avance:")
print("git add README.md")
print("git add CHANGELOG.md")
print("git add 01_Urban_Flow_Apellidos_Nombres.ipynb")
print("git add .gitignore")
print("git commit -m \"Sprint 1 - estructura inicial\"")


In [ ]:
push_si_corresponde()


## Ejercicio 02

> Puntos: 1

Se descarga el dataset original, se almacena en `urban_flow/data/raw`, y luego se inspeccionan las primeras filas, los tipos de datos y los valores nulos.


In [ ]:
urlretrieve(dataset_url, raw_dataset_path)
print(f"Dataset descargado en: {raw_dataset_path}")


In [ ]:
df_raw = pd.read_csv(raw_dataset_path)
df_raw.head()


In [ ]:
df_raw.dtypes


In [ ]:
df_raw.isna().sum()


In [ ]:
push_si_corresponde()


## Ejercicio 03

> Puntos: 2

Limpieza y transformación del dataset: normalización de fechas, horas, ubicaciones y patentes; eliminación de filas inválidas y outliers; cálculo de excesos de velocidad; y exportación del dataset limpio a `data/interim/`.


In [ ]:
FECHA_POR_DEFECTO = "1932-01-01"
HORA_POR_DEFECTO = "00:00"

def normalizar_texto(texto: object) -> str:
    if pd.isna(texto):
        return ""
    texto = str(texto).strip().upper()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(caracter for caracter in texto if not unicodedata.combining(caracter))
    texto = re.sub(r"[^A-Z0-9 ]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def normalizar_fecha(valor: object) -> str:
    if pd.isna(valor):
        return FECHA_POR_DEFECTO
    texto = str(valor).strip()
    for formato in ("%d/%m/%Y", "%Y/%m/%d", "%Y-%m-%d"):
        try:
            return datetime.strptime(texto, formato).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return FECHA_POR_DEFECTO

def normalizar_hora(valor: object) -> str:
    if pd.isna(valor):
        return HORA_POR_DEFECTO
    texto = str(valor).strip().upper()
    for formato in ("%I:%M %p", "%H:%M"):
        try:
            return datetime.strptime(texto, formato).strftime("%H:%M")
        except ValueError:
            continue
    return HORA_POR_DEFECTO

def normalizar_patente(valor: object):
    texto = normalizar_texto(valor).replace(" ", "")
    return texto if texto else pd.NA


In [ ]:
df_limpio = df_raw.copy()


In [ ]:
df_limpio["fecha"] = df_limpio["fecha"].apply(normalizar_fecha)
df_limpio["fecha"].head(10)


In [ ]:
df_limpio["hora"] = df_limpio["hora"].apply(normalizar_hora)
df_limpio["hora"].head(10)


In [ ]:
df_limpio["ubicacion"] = df_limpio["ubicacion"].apply(normalizar_texto)
df_limpio["ubicacion"].head(10)


In [ ]:
df_limpio["patente"] = df_limpio["patente"].apply(normalizar_patente)
df_limpio["patente"].tail(10)


### Eliminación de filas con valores relevantes vacíos

Se descartan las filas que no tienen los datos mínimos para considerarse una multa válida: `patente`, `velocidad_registrada` y `velocidad_maxima`.


In [ ]:
columnas_relevantes = ["patente", "velocidad_registrada", "velocidad_maxima"]

filas_antes = len(df_limpio)
df_limpio = df_limpio.dropna(subset=columnas_relevantes).reset_index(drop=True)
filas_eliminadas = filas_antes - len(df_limpio)

print(f"Se eliminaron {filas_eliminadas} filas.")
print("Las columnas observadas son:")
for columna in columnas_relevantes:
    print(f" - {columna}")


### Detección y eliminación de outliers

Se usa el rango intercuartílico (IQR) sobre las columnas numéricas de velocidad para descartar valores fuera de los límites razonables.


In [ ]:
def limites_iqr(serie: pd.Series) -> tuple[float, float]:
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

columnas_outliers = ["velocidad_registrada", "velocidad_maxima"]
mascara = pd.Series(False, index=df_limpio.index)
for columna in columnas_outliers:
    inferior, superior = limites_iqr(df_limpio[columna])
    mascara |= (df_limpio[columna] < inferior) | (df_limpio[columna] > superior)

filas_antes = len(df_limpio)
df_limpio = df_limpio.loc[~mascara].reset_index(drop=True)
filas_eliminadas = filas_antes - len(df_limpio)

print(f"Se eliminaron {filas_eliminadas} filas.")
print("Las columnas observadas son:")
for columna in columnas_outliers:
    print(f" - {columna}")


### Cálculo del exceso de velocidad real

Diferencia entre la velocidad registrada y la velocidad máxima permitida.


In [ ]:
df_limpio["exceso_velocidad_real"] = (
    df_limpio["velocidad_registrada"] - df_limpio["velocidad_maxima"]
)
df_limpio[["patente", "exceso_velocidad_real"]].head(10)


### Cálculo del exceso de velocidad con tolerancia

Se contempla un 5% de tolerancia sobre la velocidad máxima antes de considerar infracción.


In [ ]:
df_limpio["exceso_velocidad"] = (
    df_limpio["velocidad_registrada"] - df_limpio["velocidad_maxima"] * 1.05
)
df_limpio[["patente", "exceso_velocidad"]].head(10)


### Filtrado de filas sin infracción

Si `exceso_velocidad` no es positivo, la velocidad registrada queda dentro del margen tolerado y no corresponde multar.


In [ ]:
filas_antes = len(df_limpio)
df_limpio = df_limpio.loc[df_limpio["exceso_velocidad"] > 0].reset_index(drop=True)
filas_eliminadas = filas_antes - len(df_limpio)

print(f"Se eliminaron {filas_eliminadas} filas.")


### Exportación del dataset limpio

Se guarda el dataframe resultante en `urban_flow/data/interim/speeding_fines.csv` para que el próximo avance trabaje sobre datos ya depurados.


In [ ]:
interim_dataset_path = interim_dir / "speeding_fines.csv"
df_limpio.to_csv(interim_dataset_path, index=False)
print(f"Dataset limpio guardado en: {interim_dataset_path}")


In [ ]:
push_si_corresponde()


## Ejercicio 04

> Puntos: 2

Pendiente para el siguiente avance: implementación de la clase `FineAnalyzer`.


## Punto 05

> Puntos: 2

Pendiente para un avance posterior: generación y exportación de gráficos.


## Punto 06

> Puntos: 1

Pendiente para un avance posterior: cálculo de porcentajes sobre fecha y hora normalizadas.


## Punto 07

> Puntos: 1

Pendiente para un avance posterior: redacción de la conclusión final.
